# Run a Context-Engineering Experiment Yourself

Companion notebook for **Section 13, Lesson 8** of
[From Beginner to Advanced LLM Developer](https://academy.towardsai.net/courses/take/beginner-to-advanced-llm-dev).

In Lesson 7 we showed you what our experiments measured. This notebook hands you
the instrument so you can measure it yourself.

You will run a 17-turn conversation under two memory policies, check that the
experiment was actually valid, grade it offline, and read a side-by-side report.
The tool is `context-lab`, a teaching-size version of the harness behind the
production AI Tutor.

**Order matters here.** We do the free, no-API-key half first (Steps 1 to 4),
because that is the property worth internalizing: *run once, grade forever*.
Only Step 5 spends money, and it spends about three cents.

| step | needs a key? | what it does |
|---|---|---|
| 1. Install | no | |
| 2. Unpack the shipped results | no | a real run, saved to disk |
| 3. The validity gate | no | reject a run that tested nothing |
| 4. Grade and report | no | the headline result, for free |
| 5. Run it live | yes | your own numbers, ~$0.03 |
| 6. Gate, grade, report your run | no | |

## Step 1: Install

`context-lab` installs from the course notebook repository. It has three
dependencies, and two of them are only needed for the live run.

In [ ]:
# The package lives in the course notebook repo:
%pip install -q "context-lab @ git+https://github.com/towardsai/ai-tutor-rag-system.git#subdirectory=context_lab"

# Working from a local clone of the repo instead? Comment the line above and use:
# %pip install -q -e /path/to/ai-tutor-rag-system/context_lab

In [2]:
import context_lab

print("context-lab", context_lab.__version__)
print("memory configurations:", ", ".join(context_lab.CONFIGS))

context-lab 0.1.0
memory configurations: full_history, summarize, capped, summarize_prod_trigger


## Step 2: The shipped results bundle

The package ships one **pre-run results bundle**: a real run of the battery on
`gemini-3.5-flash-lite`, saved as JSON. Everything from here to Step 4 reads
those files. No key, no network, no cost.

This is not a convenience for the notebook. It is the harness's core property
made tangible. In the production program, one experiment cost about $323 to
run, and every re-analysis since has been free, because every turn was
persisted before anyone knew what questions they would want to ask of it.

`copy_prerun` copies the runs somewhere writable, because installed package
data can live in a read-only directory.

In [3]:
from context_lab import copy_prerun

run_dirs = copy_prerun("prerun_runs")
for d in run_dirs:
    print(d.name, "->", sorted(p.name for p in d.iterdir()))

full_history -> ['bundles.jsonl', 'grades_judge.jsonl', 'run_meta.json']
summarize -> ['bundles.jsonl', 'grades_judge.jsonl', 'run_meta.json']
summarize_prod_trigger -> ['bundles.jsonl', 'grades_judge.jsonl', 'run_meta.json']


Three runs, three memory configurations:

- **`full_history`** keeps every message. The control arm.
- **`summarize`** rewrites older turns into a short summary once the history
  crosses a trigger. Lossy on purpose, and it breaks the prompt cache.
- **`summarize_prod_trigger`** is the same summarization policy with a
  production-scale trigger (200,000 tokens) that a 17-turn session can never
  reach. It is here to be rejected.

Each run directory holds `run_meta.json` (the configuration) and
`bundles.jsonl` (one JSON record per turn). Let's look at one turn.

In [4]:
import json

from context_lab import load_bundles, load_meta

meta = load_meta("prerun_runs/summarize")
print("config:", json.dumps(meta["config"], indent=2))

bundles = load_bundles("prerun_runs/summarize")
turn = bundles[5]
print("\nturn", turn["turn_index"])
print("  query        :", turn["query"][:70], "...")
print("  answer chars :", len(turn["answer"]))
print("  usage        :", turn["usage"])
print("  est cost usd :", turn["est_cost_usd"])
print("  context_stats:", json.dumps(turn["context_stats"], indent=2)[:400], "...")

config: {
  "name": "summarize",
  "summarize": true,
  "summarize_trigger_tokens": 3000,
  "summarize_keep_recent_turns": 2,
  "summary_max_words": 100,
  "doc_cap_chars": null
}

turn 6
  query        : Different topic. Here are the notes on running the vector store as a s ...
  answer chars : 663
  usage        : {'input_tokens': 3200, 'output_tokens': 121, 'cached_input_tokens': None}
  est cost usd : 0.0012625
  context_stats: {
  "history_messages": 7,
  "history_tokens_approx": 3662,
  "compactions_this_turn": 1,
  "compaction_events": [
    {
      "turn_index": 6,
      "pre_compaction_tokens_approx": 3127,
      "configured_trigger_tokens": 3000,
      "summarized_messages": 3,
      "kept_messages": 4,
      "summary_chars": 610
    }
  ],
  "cap_events": []
} ...


Note `context_stats`. That block is the instrument the rest of the pipeline runs
on: how big the history was, whether compaction fired this turn, and what the
compaction event looked like. The production harness writes a wider version of
exactly this record, with real tool calls and a cache-read/cache-miss token
split.

The rule to take from this: **persist everything a future grader might need,
before you know who the grader is.**

## Step 3: The validity gate

Here is the question that separates an experiment from a number: *did the thing
you are testing actually happen?*

A memory probe asks whether a fact planted at turn 2 survived to turn 16. That
question only means something if the memory policy actually ran. If summarization
never fired, the probe measured an idle code path, and a high score is evidence
of nothing.

So the gate checks the run's own trace against its own configuration:

- a **summarizing** config must show at least one compaction, and every probe
  must land at or after the first one;
- a **non-summarizing** config must show **zero** compactions, because a control
  arm that compacted is not a control.

In [5]:
from context_lab import check_runs

results = check_runs(run_dirs, context_lab.load_battery(context_lab.default_battery_path())[0])

full_history                 config=full_history           compactions@[] probes@[12, 13, 14, 15, 16, 17] -> VALID
summarize                    config=summarize              compactions@[5, 6, 7, 10, 16] probes@[12, 13, 14, 15, 16, 17] -> VALID
summarize_prod_trigger       config=summarize_prod_trigger compactions@[] probes@[12, 13, 14, 15, 16, 17] -> INVALID: compaction never fired
gate: 2/3 runs valid (memory numbers from invalid runs must not be reported)


`summarize_prod_trigger` is rejected: **compaction never fired**.

Look at what that run would have reported if nobody checked. Its answers are
fine. Its probe score is perfect. It would have sat in a results table looking
like evidence that a 200k-token trigger preserves memory beautifully, when all
it shows is that a 17-turn session never reached the trigger and the arm was
therefore identical to `full_history`.

This is a real failure mode, not a teaching invention. In our own program, a
long-term-memory arm ran for a full battery while silently disabled: the runner
never passed a student id, so the memory middleware no-op'd, and the arm's score
was published as a finding before anyone noticed it had tested nothing.

Note that the gate is not a grader. It never looks at whether an answer is
*good*. It only asks whether the mechanism under test was engaged.

## Step 4: Grade offline, then read the report

Grading runs in two layers, cheapest first.

**Code checks** are regexes over the saved answers: does the answer name the
planted Python version, does it still use the old port after the user changed
it, does it call `print()` when the user banned `print()`. Deterministic, instant,
free, and repeatable as often as you like.

**An LLM judge** handles what regexes cannot. It is blinded: it sees the
question, the answer, and one criterion, never which configuration produced the
answer, so it cannot favor a strategy. The shipped bundle already includes judge
grades, so you do not have to pay for them now.

In [6]:
from context_lab import default_battery_path, grade_run, load_battery

battery = load_battery(default_battery_path())[0]
print(battery["description"][:400], "...\n")

for run_dir in run_dirs:
    grade_run(run_dir, battery)

Synthetic 17-turn session for a documentation-search project. Five facts are planted in turns 1-3 (Python 3.11 only; no C-extension dependencies; the ds_ collection-name prefix; type hints with logging instead of print; pandas banned from the image), one fact is updated at turn 9 (vector store port 8010 -> 9010), and turns 4-11 are filler technical Q&A, three of them carrying large document payloa ...

full_history: code-graded 6 probes, 6 pass
summarize: code-graded 6 probes, 5 pass
summarize_prod_trigger: code-graded 6 probes, 6 pass  [gate: INVALID: compaction never fired]


Every grade row carries provenance: who graded it, when, and whether the run
passed the gate. In production this mattered more than it sounds. One battery
was graded by two different judges after an interrupted pass, and the mixed
provenance made one arm read 63% instead of 98%. The rule that came out of it:
**one battery gets exactly one judge, and grade provenance is part of the
measurement.**

In [7]:
from context_lab import load_grades

for run_dir in run_dirs:
    print(f"\n{run_dir.name}")
    for row in load_grades(run_dir, "grades_code.jsonl"):
        flag = "" if row["gate_valid"] else "   <- run failed the gate"
        print(f"  t{row['turn_index']:2d} {row['item_type']:30s} {row['grade']:4s}{flag}")


full_history
  t12 probe:fact_recall              pass
  t13 probe:fact_update              pass
  t14 probe:preference_compliance    pass
  t15 probe:fact_recall              pass
  t16 probe:fact_recall              pass
  t17 probe:preference_compliance    pass

summarize
  t12 probe:fact_recall              pass
  t13 probe:fact_update              pass
  t14 probe:preference_compliance    pass
  t15 probe:fact_recall              pass
  t16 probe:fact_recall              fail
  t17 probe:preference_compliance    pass

summarize_prod_trigger
  t12 probe:fact_recall              pass   <- run failed the gate
  t13 probe:fact_update              pass   <- run failed the gate
  t14 probe:preference_compliance    pass   <- run failed the gate
  t15 probe:fact_recall              pass   <- run failed the gate
  t16 probe:fact_recall              pass   <- run failed the gate
  t17 probe:preference_compliance    pass   <- run failed the gate


Now the side-by-side report. Rows are metrics, columns are configurations, which
makes "what did summarizing cost me?" a single horizontal read.

In [8]:
from IPython.display import Markdown

from context_lab import build_report

Markdown(build_report(run_dirs, battery))

# context-lab report

battery: `docsearch_v1` (17 turns, 6 probes)
model: `gemini-3.5-flash-lite` via `gemini`

Token and cost rows are the efficiency headline. Latency rows are marked
`[confounded]`: arms run sequentially against a shared API, so wall-clock
differences include provider load. Memory rows read `n/a (gate)` when the run
failed the validity gate, because a memory number from a run where compaction
never fired measures an idle code path.

**Read the cache hit ratio row before you read the cost rows.** Keeping the
full history is cheap only when the repeated prefix is billed at a cache
discount. `not reported` means the provider returned no cache accounting, so
every input token here was priced at the full rate and the cost comparison
below is a no-cache comparison. That is a different experiment from the one
a cache-discounted provider runs, and its cost answer can legitimately flip.

| metric | full_history | summarize | summarize_prod_trigger |
|---|---|---|---|
| validity gate | VALID | VALID | **INVALID** |
| gate reason | - | - | compaction never fired |
| turns | 17 | 17 | 17 |
| errors | 0 | 0 | 0 |
| compactions fired | 0 | 5 | 0 |
| payloads capped | 0 | 0 | 0 |
| input tokens (billed, all calls) | 95,884 | 35,071 | 88,258 |
| output tokens | 4,868 | 4,747 | 3,784 |
| cache hit ratio (input) | not reported | not reported | not reported |
| final history size (approx tokens) | 9,058 | 845 | 8,205 |
| est cost, whole session | $0.0409 | $0.0270 | $0.0359 |
| est cost / turn | $0.0024 | $0.0016 | $0.0021 |
| of which summarization | $0.0000 | $0.0046 | $0.0000 |
| median latency ms [confounded] | 1443 | 1170 | 924 |
| probe accuracy [code] | 100% (6/6) | 83% (5/6) | n/a (gate) |
| probe accuracy [judge] | 100% (6/6) | 83% (5/6) | n/a (gate) |
| └ fact_recall [code] | 100% (3/3) | 67% (2/3) | n/a (gate) |
| └ fact_update [code] | 100% (1/1) | 100% (1/1) | n/a (gate) |
| └ preference_compliance [code] | 100% (2/2) | 100% (2/2) | n/a (gate) |


### Reading the report

Three things to notice, in order.

**The gate is a row, not a footnote.** `summarize_prod_trigger` shows
`**INVALID**`, and every memory metric in its column reads `n/a (gate)`. Its cost
and token numbers are still printed, because those dollars were really spent.
Only the claims the run cannot support are withheld. An invalid memory number
should be hard to quote by accident.

**The memory result reproduces the lesson.** `full_history` scores 100% on the
probes. `summarize` loses ground, and the sub-rows show *which* memory it lost:
`fact_update` survives, because the update happened at turn 9 and sits inside the
window summarization keeps verbatim. What degrades is the material planted in the
first three turns, and which of those facts is lost varies from run to run. That
is the same shape our production sessions showed, where compaction arms kept 100%
of mid-session updates and collapsed on turn-0 material.

**The cost result does not reproduce, and the report tells you why.** Read the
`cache hit ratio (input)` row: `not reported`. On this model the API returned no
cache accounting at all, so every input token was billed at the full rate. Under
those conditions the arm that sends fewer tokens is simply cheaper, and
`summarize` costs less per turn than `full_history`.

That last point is the most useful thing in this notebook, so we will not rush
past it.

### Why "summarizing is more expensive" did not show up here

Our headline finding was that keeping the full history was *cheaper* than
summarizing. That result rests entirely on **prompt caching**. When the start of
your prompt is byte-identical to the previous turn, providers bill those tokens
at a steep discount. Keeping the history intact preserves that prefix.
Summarizing rewrites it, throwing the cache away and re-paying full price for
everything.

The size of the discount decides the winner:

| provider | cached input discount |
|---|---|
| DeepSeek V4 Flash | about 50x cheaper |
| Gemini 3.5 Flash | about 10x cheaper |
| no cache engagement | no discount at all |

This run got no discount, so the economics inverted. That is not a bug in the
harness, and it is not a refutation of the finding. It is the finding's
precondition, made visible.

The engineering habit worth building: **read the cache hit ratio before you read
the cost column.** A cost comparison without caching is a different experiment
from a cost comparison with it, and the two can give opposite answers. Notice
that we could only tell the difference because the report distinguishes
`not reported` from `0%`. A harness that printed `0%` for missing data would
have quietly asserted a measurement nobody took.

## Step 5: Run it yourself

Everything so far was free. This step calls a real API.

The default is Gemini `gemini-3.5-flash-lite`. The whole run below is three
configurations over 17 turns, which measured at **about $0.03** total. Set your
key and run it.

**On providers:** this notebook deliberately sits outside the course's
three-provider pattern. Context-engineering economics are cache-discount
economics, and cache discounts differ per provider, so "pick any of three" would
change the answer rather than just the vendor. Your options:

- **Free:** a local model through [Ollama](https://ollama.com) (`provider="ollama"`).
  Costs nothing. Our small-model runs used 7-8B models at a 32k window on a 16 GB
  laptop.
- **Cheap:** DeepSeek (`provider="deepseek"`), fractions of a cent per turn, and
  the deepest cache discount of the three, so it is the best place to see the
  caching effect.
- **Tested here:** Gemini.

The DeepSeek and Ollama backends are written and configured but were not
exercised by this notebook's executed run. Treat them as starting points.

In [9]:
import os

# In Colab:
# from google.colab import userdata
# os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

assert os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY"), (
    "Set GEMINI_API_KEY to run this step. Steps 1-4 above need no key."
)
print("key found")

key found


In [10]:
from context_lab import CONFIGS, get_provider, run_session

provider = get_provider("gemini")  # or "deepseek" / "ollama"

my_runs = []
for name in ["full_history", "summarize", "summarize_prod_trigger"]:
    print(f"\n=== {name} ===")
    my_runs.append(run_session(battery, CONFIGS[name], provider, f"my_runs/{name}",
                               battery_path=default_battery_path()))


=== full_history ===


  turn  1 [full_history] 138 in / 2.2s


  turn  2 [full_history] 610 in / 2.0s


  turn  3 [full_history] 1,096 in / 2.3s


  turn  4 [full_history] 2,922 in / 2.4s


...

  turn 13 [summarize_prod_trigger] 9,154 in / 0.6s


  turn 14 [summarize_prod_trigger] 9,206 in / 1.3s


  turn 15 [summarize_prod_trigger] 9,539 in / 1.1s


  turn 16 [summarize_prod_trigger] 9,661 in / 0.6s


  turn 17 [summarize_prod_trigger] 9,711 in / 1.9s


Each line shows the billed input tokens for that turn, the latency, and a
`[compacted]` marker when the memory policy fired.

Watch the input-token column across the three arms. Under `full_history` it
climbs every turn, because the whole conversation is re-sent and re-billed each
time. Under `summarize` it climbs, drops when compaction fires, and climbs
again. That sawtooth *is* context engineering, visible in the billing.

## Step 6: Gate, grade, and report your own run

Same three commands, now against your own bundles. All free.

In [11]:
my_results = check_runs(my_runs, battery)

full_history                 config=full_history           compactions@[] probes@[12, 13, 14, 15, 16, 17] -> VALID
summarize                    config=summarize              compactions@[5, 6, 7, 10] probes@[12, 13, 14, 15, 16, 17] -> VALID
summarize_prod_trigger       config=summarize_prod_trigger compactions@[] probes@[12, 13, 14, 15, 16, 17] -> INVALID: compaction never fired
gate: 2/3 runs valid (memory numbers from invalid runs must not be reported)


In [12]:
for run_dir in my_runs:
    grade_run(run_dir, battery)

full_history: code-graded 6 probes, 6 pass
summarize: code-graded 6 probes, 5 pass
summarize_prod_trigger: code-graded 6 probes, 6 pass  [gate: INVALID: compaction never fired]


In [13]:
Markdown(build_report(my_runs, battery))

# context-lab report

battery: `docsearch_v1` (17 turns, 6 probes)
model: `gemini-3.5-flash-lite` via `gemini`

Token and cost rows are the efficiency headline. Latency rows are marked
`[confounded]`: arms run sequentially against a shared API, so wall-clock
differences include provider load. Memory rows read `n/a (gate)` when the run
failed the validity gate, because a memory number from a run where compaction
never fired measures an idle code path.

**Read the cache hit ratio row before you read the cost rows.** Keeping the
full history is cheap only when the repeated prefix is billed at a cache
discount. `not reported` means the provider returned no cache accounting, so
every input token here was priced at the full rate and the cost comparison
below is a no-cache comparison. That is a different experiment from the one
a cache-discounted provider runs, and its cost answer can legitimately flip.

| metric | full_history | summarize | summarize_prod_trigger |
|---|---|---|---|
| validity gate | VALID | VALID | **INVALID** |
| gate reason | - | - | compaction never fired |
| turns | 17 | 17 | 17 |
| errors | 0 | 0 | 0 |
| compactions fired | 0 | 4 | 0 |
| payloads capped | 0 | 0 | 0 |
| input tokens (billed, all calls) | 95,524 | 40,036 | 108,440 |
| output tokens | 4,909 | 4,866 | 6,243 |
| cache hit ratio (input) | not reported | not reported | not reported |
| final history size (approx tokens) | 9,276 | 3,003 | 10,420 |
| est cost, whole session | $0.0409 | $0.0279 | $0.0481 |
| est cost / turn | $0.0024 | $0.0016 | $0.0028 |
| of which summarization | $0.0000 | $0.0037 | $0.0000 |
| median latency ms [confounded] | 1983 | 1048 | 1768 |
| probe accuracy [code] | 100% (6/6) | 83% (5/6) | n/a (gate) |
| └ fact_recall [code] | 100% (3/3) | 100% (3/3) | n/a (gate) |
| └ fact_update [code] | 100% (1/1) | 100% (1/1) | n/a (gate) |
| └ preference_compliance [code] | 100% (2/2) | 50% (1/2) | n/a (gate) |


Your numbers will not match the shipped bundle exactly. The model is sampling,
compaction fires at different turns depending on how long the answers ran, and a
single 17-turn session with six probes is a small sample. Across four runs of this exact battery, `summarize` scored between 50% and 83%
while `full_history` stayed at 100% every time.

Treat one run as an anecdote. The production screens ran 1 to 3 trials per cell
and still describe their confidence carefully. If a difference matters to you,
run it again.

## Exercises

**1. Cap the payloads instead of summarizing.** The battery attaches three large
documents to user turns, standing in for tool outputs. The `capped`
configuration truncates them **once**, when they enter the history, and never
rewrites them again. That keeps the cached bytes identical every turn, which is
exactly why it behaves differently from summarization.

```python
run_session(battery, CONFIGS["capped"], provider, "my_runs/capped",
            battery_path=default_battery_path())
```

Add it to the report. In our production experiments this idea, applied as a
stable cap, cut cost 38% with no measured quality or memory loss, while the
same idea applied as *per-call* truncation made things worse: the agent kept
re-calling tools to recover what had been cut.

**2. Break the gate on purpose.** Raise `summarize_trigger_tokens` on the
`summarize` config until compaction stops firing, then re-run the gate. Confirm
that the run is rejected and that its memory column blanks out.

**3. Find a code-check false negative.** The code checks are regexes, and
regexes are proxies. Read the answers next to their grades and look for one the
checks got wrong. We found two while building this battery: a correct answer
that wrote `requires-python = "==3.11.*"` and then explained which versions that
blocks, and a good answer that proposed a non-`pandas` library the checks had
not anticipated. Both are recorded as comments in the battery source. This is
the habit the lesson is really teaching: **a metric that disagrees with the
answers is a bug in the metric until proven otherwise.**

**4. Write a fourth probe type.** The battery probes `fact_recall`,
`fact_update`, and `preference_compliance`. Add a `contradiction` probe: state a
fact, contradict it later, and check which one the model believes.

**5. Run it on DeepSeek** and compare the `cache hit ratio` row against what you
saw here. That is where the cost story becomes visible.

## What to take away

- Every turn is persisted, so **grading and reporting cost nothing and can be
  repeated forever**. Only running spends money.
- **The validity gate comes before the grade.** A number from a run where the
  mechanism never fired is worse than no number, because it looks like evidence.
- **Memory degradation is real and it is selective.** Summarization keeps what
  is recent and drops what was established early.
- **Cost conclusions are provider-conditional.** They live or die on the cache
  discount, so check the cache row before you compare dollars.

Lesson 9 opens the next section. If you want to go deeper on the harness itself,
Lesson 6 covers how the production version was built and Lesson 7 covers what it
measured.